# 📓 Semana 10 · Dia 3 — ML lifecycle: modelo de previsão de vendas

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | MLP, MLA |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Modelo treinado e avaliado no MLflow |

---


## 📖 Teoria — O modelo do projeto

Vamos prever a **receita diária** com base em features temporais (dia da semana, mês, ano, feriado). Isso exercita o fluxo real de ML: features → treino → avaliação → registro.


### 💻 Na prática — Features + treino

Prepare features e treine o modelo com busca simples de hiperparâmetros.


In [ ]:
# Features de calendário a partir do Ouro
from pyspark.sql.functions import dayofweek, month, year, dayofmonth, lag as lagf
from pyspark.sql.window import Window
df = (spark.table("workspace.ouro.vendas_por_dia")
    .withColumn("dia_semana", dayofweek("data_venda"))
    .withColumn("dia_mes", dayofmonth("data_venda"))
    .withColumn("mes", month("data_venda"))
    .withColumn("ano", year("data_venda"))
    .toPandas())
print(df.shape)

In [ ]:
# Treino com busca de hiperparâmetros (2 runs)
import mlflow
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

X = df[["dia_semana", "dia_mes", "mes", "ano"]]
y = df["receita_total"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

mlflow.autolog()
with mlflow.start_run(run_name="gbm_vendas"):
    modelo = GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.1)
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    mlflow.log_metrics({"rmse": mean_squared_error(y_test, pred)**0.5,
                        "mae": mean_absolute_error(y_test, pred),
                        "r2": r2_score(y_test, pred)})
    print("Run GBM registrada.")

### 💻 Na prática — Comparação e baseline

Na UI: compare as runs; uma métrica de baseline (ex.: prever sempre a média) mostra se o modelo agrega valor.


In [ ]:
# Baseline: prever a média (regra simples)
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
baseline = np.full_like(y_test, y_train.mean())
print("Baseline RMSE:", mean_squared_error(y_test, baseline)**0.5)
print("R2 do baseline (deve ser <= 0):", r2_score(y_test, baseline))
print("Se o modelo ML tiver R2 > 0 e RMSE < baseline, ele agrega valor.")

> 🎯 **Dica de prova**: MLP: saber ler métricas (RMSE/MAE/R2) e comparar com baseline é pergunta garantida. R2 > 0 já indica ganho sobre a média.


## 🎯 Exercícios de fixação

**1.** Registre o melhor modelo com alias champion.

**2.** O que significaria um R2 negativo?

**3.** Treine com uma feature extra (ex.: lag da receita) e compare.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Registrar champion

`mlflow.register_model('runs:/<run_id>/model', 'workspace.prata.modelo_previsao_receita')` + set alias champion na versão nova.

**2.** R2 negativo

O modelo prevê PIOR que a média — sinal de features ruins ou overfit. Volte às features.

**3.** Lag feature

Adicione `lag(receita, 7)` (janela de 7 dias) via Window no Spark; se o R2 subir, a sazonalidade semanal explica parte da receita.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*